In [0]:
import pandas as pd

event_identifier = pd.DataFrame({
    "event_type": [1, 2, 3, 4, 5],
    "event_name": ["Page View", "Add to Cart", "Purchase", "Ad Impression", "Ad Click"]
})

event_identifier


In [0]:
campaign_identifier = pd.DataFrame({
    "campaign_id": [1, 2, 3],
    "products": ["1-3", "4-5", "6-8"],
    "campaign_name": [
        "BOGOF - Festival Deals",
        "25% Off - Wedding Essentials",
        "Half Off - New Year Bonanza"
    ],
    "start_date": pd.to_datetime(["2020-01-01", "2020-01-15", "2020-02-01"]),
    "end_date": pd.to_datetime(["2020-01-14", "2020-01-28", "2020-03-31"])
})

campaign_identifier


In [0]:
page_hierarchy = pd.DataFrame({
    "page_id": [1,2,3,4,5,6,7,8,9,10,11,12,13],
    "page_name": [
        "Home Page", "All Products", "Men’s Kurta Collection",
        "Sarees & Lehengas", "Casual Footwear", "Designer Handbags",
        "Gold Plated Jewelry", "Smartphones", "Laptops",
        "Kitchen Appliances", "Decor & Furnishings",
        "Checkout", "Confirmation"
    ],
    "product_category": [
        None, None, "Ethnic Wear", "Ethnic Wear", "Footwear",
        "Accessories", "Accessories", "Electronics", "Electronics",
        "Home & Kitchen", "Home & Kitchen", None, None
    ],
    "product_id": [
        None, None, 1, 2, 3, 4, 5, 6, 7, 8, 9, None, None
    ]
})

page_hierarchy


In [0]:
users = pd.DataFrame({
    "user_id": [1, 2, 3],
    "cookie_id": ["A1B2C3", "D4E5F6", "G7H8I9"],
    "start_date": pd.to_datetime(["2020-01-01", "2020-01-02", "2020-01-03"])
})

users


In [0]:
import pandas as pd

events = pd.DataFrame({
    "visit_id": [1, 1, 1, 2, 2, 3],
    "cookie_id": ["A1B2C3", "A1B2C3", "A1B2C3",
                  "D4E5F6", "D4E5F6",
                  "G7H8I9"],
    "page_id": [1, 3, 12, 2, 5, 8],
    "event_type": [1, 2, 3, 1, 2, 4],
    "sequence_number": [1, 2, 3, 1, 2, 1],
    "event_time": pd.to_datetime([
        "2020-01-01 10:00:00",
        "2020-01-01 10:05:00",
        "2020-01-01 10:10:00",
        "2020-01-02 11:00:00",
        "2020-01-02 11:08:00",
        "2020-01-03 12:00:00"
    ])
})

events


### # event_identifier → event_type, event_name
### # campaign_identifier → campaign_id, products, campaign_name, start_date, end_date
### # page_hierarchy → page_id, page_name, product_category, product_id
### # users → user_id, cookie_id, start_date
### # events → visit_id, cookie_id, page_id, event_type, sequence_number, event_time

In [0]:
# SET A
# 1 How many distinct users are in the dataset?
distinct_user=users["user_id"].nunique()
distinct_user

In [0]:
# 2 What is the average number of cookie IDs per user?
avg_cookie = users.groupby("user_id")["cookie_id"].nunique().mean()
avg_cookie


In [0]:
#3  What is the number of unique site visits by all users per month?
user_event=users.merge(events,on="cookie_id",how="left")
user_event["month"]=user_event["event_time"].dt.month
user_event_unique=user_event.groupby("month")["visit_id"].nunique().reset_index()
user_event_unique

In [0]:
#4 What is the count of each event type?
events_ide=events.merge(event_identifier,on="event_type",how="left")
events_ide_count=events_ide.groupby("event_name").size().reset_index()
events_ide_count


In [0]:
# 5 What percentage of visits resulted in a purchase?
import numpy as np
events_ide_total=events_ide["visit_id"].nunique()
events_ide_purchase=events_ide[events_ide["event_type"] == 3]["visit_id"].nunique()
percentage=events_ide_purchase / events_ide_total *100
percentage

In [0]:
#6 What percentage of visits reached checkout but not purchase?
events_ide_total = events_ide["visit_id"].nunique()
checkout_visits = events_ide[events_ide["page_id"] == 12]["visit_id"].unique()
purchase_visits = events_ide[events_ide["event_name"] == "Purchase"]["visit_id"].unique()
checkout_not_purchase_visits = np.setdiff1d(checkout_visits, purchase_visits)
percentage = len(checkout_not_purchase_visits) / events_ide_total * 100
percentage


In [0]:
#7 What are the top 3 most viewed pages?
events_ides = events_ide[events_ide["event_type"]==1]
events_ides =events_ides.groupby("page_id").size().sort_values(ascending=False).head(3).reset_index(name="count")
events_ides

In [0]:
#8 What are the views and add-to-cart counts per product category?
events_ide_page=events_ide.merge(page_hierarchy,on="page_id",how="left")
filtered=events_ide_page[events_ide_page["event_type"].isin([1,2])]
result=filtered.groupby(["product_category","event_type"]).size().unstack(fill_value=0)
result


In [0]:
#9 What are the top 3 products by purchases?
events_purchases=events_ide_page[events_ide_page["event_type"]==3] \
    .groupby("product_id").size().sort_values(ascending=False).head(3).reset_index()

events_purchases

In [0]:
events_ide_page

In [0]:
# SET B
#10  Create a product-level funnel table with views, cart adds, abandoned carts, and purchases.
def funnel_calc(group):
    return pd.Series({
        "views": (group["event_type"] == 1).sum(),
        "cart": (group["event_type"] == 2).sum(),
        "purchase": (group["event_type"] == 3).sum()})
funnel = events_ide_page.groupby("product_id").apply(funnel_calc)
funnel


In [0]:
def funnel_Cal(group):
    return pd.Series({
        "views": (group["event_type"]==1).sum(),
        "cart" : (group["event_type"] == 2).sum() ,
        "purchase" : (group["event_type"] == 3).sum()
    })
funnel=events_ide_page.groupby("product_id").apply(funnel_Cal)
funnel

In [0]:
#11 Create a category-level funnel table with the same metrics as above.
funnel = events_ide_page.groupby("product_category").agg(
    views=("event_type", lambda x: (x==1).sum()),
    cart=("event_type", lambda x: (x==2).sum()),
    purchase=("event_type", lambda x: (x==3).sum())
)
funnel

In [0]:
12 # Which product had the most views, cart adds, and purchases?
funnel = events_ide_page.groupby("product_id").agg(
    views=("event_type", lambda x: (x==1).sum()),
    cart=("event_type", lambda x: (x==2).sum()),
    purchase=("event_type", lambda x: (x==3).sum())
)

most_viewed = funnel.loc[[funnel["views"].idxmax()]]
most_cart = funnel.loc[[funnel["cart"].idxmax()]]
most_purchase = funnel.loc[[funnel["purchase"].idxmax()]]
print("Most Viewed Product:")
print(most_viewed)
print("\nMost Cart Added Product:")
print(most_cart)
print("\nMost Purchased Product:")
print(most_purchase)

In [0]:
#13 Which product was most likely to be abandoned?
# 1️⃣ Total cart adds
# 2️⃣ Total purchases
# 3️⃣ Abandoned carts = cart - purchase
# 4️⃣ Abandonment rate = (cart - purchase) / cart

total_cart=events_ide_page[events_ide_page["event_name"]=="Add to Cart"].groupby("product_id").size()
total_purchases=events_ide_page[events_ide_page["event_name"]=="Purchase"].groupby("product_id").size()
abandoned_cart = total_cart.subtract(total_purchases, fill_value=0)
abandon_rate = abandoned_cart / total_cart
most_abandoned_product = abandon_rate.idxmax()
most_abandoned_product

In [0]:
# 14 Which product had the highest view-to-purchase conversion rate?
# total_view , total_purchase,conversion_rate=total_purchase/total_view
events_ide_page
total_view=events_ide_page[events_ide_page["event_name"]=="Page View"].groupby("product_id").size()
total_purchases=events_ide_page[events_ide_page["event_name"]=="Purchase"].groupby("product_id").size()
conversion_Rate=total_purchases.divide(total_view,fill_value=(0))
conversion_Rate

In [0]:

# 15 What is the average conversion rate from view to cart add?
total_view=events_ide_page[events_ide_page["event_name"]=="Page View"].groupby("product_id").size()
total_cart=events_ide_page[events_ide_page["event_name"]=="Add to Cart"].groupby("product_id").size()
conversion_rate_avg = (
    total_cart.divide(total_view, fill_value=0)
    .mean()
)
conversion_rate_avg

In [0]:
#16 What is the average conversion rate from cart add to purchase?
total_cart=events_ide_page[events_ide_page["event_name"]=="Add to Cart"].groupby("product_id").size()
total_purchase=events_ide_page[events_ide_page["event_name"]=="Purchase"].groupby("product_id").size()
conversion_rate_avg = (
    total_purchase.divide(total_cart, fill_value=0)
    .mean()
)
conversion_rate_avg

In [0]:
# SET C.
# Create a visit-level summary table with user_id, visit_id, visit start time, event counts, and campaign name.


In [0]:
# (Optional) Add a column for comma-separated cart products sorted by order of addition.


In [0]:
# Further Investigations
# Identify users exposed to campaign impressions and compare metrics with those who were not.


In [0]:
# Does clicking on an impression lead to higher purchase rates?


In [0]:
# What is the uplift in purchase rate for users who clicked an impression vs. those who didn’t?


In [0]:
# What metrics can be used to evaluate the success of each campaign?